In [20]:
# flat_logit_pymc.py  – numerically safe baseline
import pymc as pm
import pytensor.tensor as pt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# 1. data ------------------------------------------------------------
feats = pd.read_csv("../../dataset/match_features_merged.csv")
labels = pd.read_csv("../../cleaned_data/final_dataset.csv",
                     usecols=["game_id", "result"])
df = feats.merge(labels, on="game_id")

# ----  **drop unknown labels BEFORE mapping** -------------------------
label_map = {"Home Win": 0, "Draw": 1, "Away Win": 2}

# keep only rows whose label is one of the three
df = df[df["result"].isin(label_map)]

y = df["result"].map(label_map).astype("int8").values
print("unique y values:", np.unique(y))            # must be [0 1 2]
assert np.isin(y, [0, 1, 2]).all()

# --------------- 1. build matrix safely -----------------
drop_cols = ["game_id", "result"]
X_df = (
    df.drop(columns=drop_cols)
      .replace([np.inf, -np.inf], np.nan)   # get rid of inf
      .fillna(df.mean(numeric_only=True))   # simple mean-impute
)

X = X_df.astype(float).values
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_tr)
X_tr = scaler.transform(X_tr)
X_te = scaler.transform(X_te)

# --- AFTER scaling – hard-kill all non-finite numbers -------------
X_tr = np.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)
X_te = np.nan_to_num(X_te, nan=0.0, posinf=0.0, neginf=0.0)

assert np.isfinite(X_tr).all(), "still got bad values!"

print("shape X_tr:", X_tr.shape)
print("unique y_tr:", np.unique(y_tr), "dtype:", y_tr.dtype)
print("any non-finite in X_tr?", ~np.isfinite(X_tr).any())
print("min / max of X_tr:", X_tr.min(), X_tr.max())

n, k = X_tr.shape
n_cat = 3

# 2. model -----------------------------------------------------------
with pm.Model() as flat_logit:
    X_shared = pm.Data("X", X_tr)        # << wrap as shared variable

    # tighter priors keep start log-odds moderate
    beta = pm.Normal("beta", 0.0, 1.0, shape=(k, n_cat - 1))
    intercept = pm.Normal("intercept", 0.0, 1.0, shape=(n_cat - 1,))

    eta = intercept + pt.dot(X_shared, beta)        # shape (n,2)
    eta_full = pt.concatenate([pt.zeros((n, 1)), eta], axis=1)
    p = pm.math.softmax(eta_full)

    pm.Categorical("MatchOutcome", p=p, observed=y_tr)

    # ---------- 2-B  find a “safe” starting point ----------
    start = pm.find_MAP()                       # deterministic MAP
    trace = pm.sample(
        1_000, tune=1_000,
        initvals=start,                          # <-- use the MAP
        init="jitter+adapt_diag",                # small noise around it
        target_accept=0.9,
        chains=2,
    )

# 3. posterior-predictive on test set -------------------------------
with flat_logit:
    X_shared.set_value(X_te)              # swap in test matrix
    ppc = pm.sample_posterior_predictive(trace, var_names=["MatchOutcome"])

y_pred = ppc["MatchOutcome"].mode(axis=0).astype(int)  # most frequent draw

print(classification_report(y_te, y_pred, digits=3))

unique y values: [0 1 2]
shape X_tr: (47684, 53)
unique y_tr: [0 1 2] dtype: int8
any non-finite in X_tr? False
min / max of X_tr: -32.78929207319025 51.084227166034864


SamplingError: Initial evaluation of model at starting point failed!
Starting values:
{'beta': array([[0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.]]), 'i

In [ ]:
import pymc
import sys
print("PyMC", pymc.__version__)

PyMC 5.18.0
